> **Note:** This notebook requires a live OpenAI API key. Set `OPENAI_API_KEY` in your `.env` file before running. Smoke-run deferred — API key not available in CI.

In [ ]:
%pip install openai python-dotenv

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

# OpenAI Function (Tool) Calling — Responses API

OpenAI lets you connect models to outside tools. With the **Responses API** you pass tool definitions via `tools=[{...}]` and control invocation with `tool_choice`. Function tools use a **flat** shape (`type`, `name`, `description`, `parameters`) — note this differs from the older Chat Completions nesting under a `function` key.

See OpenAI's [function calling guide](https://platform.openai.com/docs/guides/function-calling).

Let's implement a simple `create_directory()` function as a tool the model can call:

In [1]:
import json
import subprocess

def create_directory(directory_name):
    """Function that creates a directory given a directory name."""
    subprocess.run(["mkdir", directory_name])
    return json.dumps({"directory_name": directory_name})


# Responses API function tool — note the FLAT shape (no nested "function" key).
tool_create_directory = {
    "type": "function",
    "name": "create_directory",
    "description": "Create a directory given a directory name.",
    "parameters": {
        "type": "object",
        "properties": {
            "directory_name": {
                "type": "string",
                "description": "The name of the directory to create.",
            }
        },
        "required": ["directory_name"],
    },
}

tools = [tool_create_directory]

The Responses API returns tool calls as items in `response.output`. We execute the function, feed the result back via a `function_call_output` item, and chain with `previous_response_id` to get the final answer.

In [4]:
def run_terminal_task():
    input_messages = [
        {"role": "user", "content": "Create a folder called questions-from-students."}
    ]
    response = client.responses.create(
        model="gpt-5.5",
        input=input_messages,
        tools=tools,
    )

    available_functions = {"create_directory": create_directory}

    # Collect tool-call outputs to send back to the model.
    tool_outputs = []
    for item in response.output:
        if item.type == "function_call":
            fn = available_functions[item.name]
            args = json.loads(item.arguments)
            result = fn(directory_name=args.get("directory_name"))
            tool_outputs.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": result,
            })

    if tool_outputs:
        second_response = client.responses.create(
            model="gpt-5.5",
            previous_response_id=response.id,
            input=tool_outputs,
        )
        return second_response
    return response


output = run_terminal_task()
print(output.output_text)

Created the folder `questions-from-students`.


In [5]:
!ls -d */ | grep students

questions-from-students/


Great! We implemented function calling with the Responses API. See OpenAI's [cookbook](https://cookbook.openai.com/examples/how_to_call_functions_with_chat_models) for more examples.

## Combining a custom function tool with the hosted `web_search` tool

The Responses API also ships **hosted tools** like `web_search`. You can mix a hosted tool with your own function tool in the same `tools` list and let the model pick.

In [6]:
# Hosted web_search tool alongside our custom function tool.
mixed_tools = [
    tool_create_directory,
    {"type": "web_search"},
]

response = client.responses.create(
    model="gpt-5.5-mini",
    input="What are the latest headlines about large language models today?",
    tools=mixed_tools,
)
print(response.output_text)

Here are the **top large-language-model headlines I’m seeing today, Thursday, June 11, 2026**:

- **Anthropic launches “Claude Corps” for nonprofits.** Anthropic is donating **$150 million** to create a fellowship program that will embed **1,000 Claude-trained fellows** in nonprofits for a year to help them use AI more effectively. ([apnews.com](https://apnews.com/article/b1c130a08417d13e1256f8982d233b0e?utm_source=openai))

- **Anthropic pledges $200 million to study AI’s economic impact.** The company announced funding for research into AI’s effects on jobs and the economy, alongside new policy proposals from CEO Dario Amodei about support for people financially affected by AI-driven job disruption. ([apnews.com](https://apnews.com/article/afeb5279eef406980dffa46ff91495e0?utm_source=openai))

- **Anthropic’s Claude Fable 5 / Mythos 5 launch is still dominating LLM news.** Anthropic says **Claude Fable 5** is a “Mythos-class” model made safe for general use, while **Claude Mythos 5** 